# WORKSHOP 2: SILVER COVER — CLEANING AND ENRICHING

**Lakehouse** : `Lakehouse_silver` · **Notebook** : `NB_02_Silver`
**Objective**: clean, standardize, enrich and structure Bronze data for analysis.

| Aspect | Bronze | Silver |
| --- | --- | --- |
| State | Crude, heterogeneous, with errors | cleaned, validated, standardized |
| Granularity | 15 min (18 144 lines) | time (4 320 lines) |
| Dates | formats multiples | one`timestamp` |
| Enrichment | none | EPEX price, benchmark site, weather, baseline, anomalies |

## Prerequisite

1. Workshop **Bronze**carried out (`Lakehouse_bronze.bronze.*`,`consumption_raw`= 18,144 lines)
2. Lakehouse **`Lakehouse_silver`** created, this notebook attached with:
   - `Lakehouse_silver`in lakehouse **by default** (read/write)
   - `Lakehouse_bronze`in lakehouse **secondary** (reading)

## Pipeline

```
Bronze (18 144) → DISTINCT → Dedup (17 280) → Filtre (NULL, <0, ≥10) → Clean (16 426)
→ Dates normalisées (16 426) → Agrégation 1 h → Hourly (4 320) → JOIN prix → With_prices (4 320)
→ Window functions → Enriched (4 320) → Prédiction pondérée → Predictions (4 320)
```

---
## Cell 0: Initial Exploration from Bronze

- **But**: check that Bronze is **readable** and see a sample
- **Qualified name**:`Lakehouse_bronze.bronze.consumption_raw`= lakehouse + diagram + table

In [ ]:
df = spark.sql("SELECT * FROM Lakehouse_bronze.bronze.consumption_raw")
df.show(10)
print(f"📊 Données Bronze : {df.count()} lignes")

**Intended**:

- **Volume** : `18144`lines = 17,280 theoretical + **864 duplicates**
- **Colonnes** : `timestamp`, `site_id`, `consumption_mw`, `voltage_v`, `frequency_hz`, `status` (`OK` / `ERROR`)
- **Visible defects**: duplicates, null, negative, outside, **multiple format dates**

---
## Cell 1: Creating the schema`silver`

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS Lakehouse_silver.silver")
print('Le schéma est prêt')

**Intended**:

- **Message** : `Le schéma est prêt`
- **DDL ideal** : no error if the schema already exists
- **Destination**: the following tables will go in`Lakehouse_silver.silver.*`

---
## Cell 2: Splitting (`consumption_dedup`)

- **`SELECT DISTINCT *`**: keep one copy** of the same lines on all columns
- **Why**: Duplicates distort averages and counts

In [ ]:
df_dedup = spark.sql("SELECT DISTINCT * FROM Lakehouse_bronze.bronze.consumption_raw")
df_dedup.write.mode("overwrite").format("delta").saveAsTable("Lakehouse_silver.silver.consumption_dedup")
print(f"✅ Table silver.consumption_dedup créée : {df_dedup.count()} lignes")

**Intended**:

- **Released**:`17280`lines
- **Deleted**: **864** exact duplicates (4.8%)

---
## Cell 3: Filtering invalid values (`consumption_clean`)

- **NULL** : consommation absente, inutilisable
- **Negative**: **error codes** (−777, -888, -999)
- **Outlets (≥ 10 MW)**: beyond the maximum capacity of a site (

In [ ]:
from pyspark.sql import functions as F

df_clean = spark.table("Lakehouse_silver.silver.consumption_dedup").filter(
    (F.col("consumption_mw").isNotNull()) &  # Pas de NULL
    (F.col("consumption_mw") >= 0) &         # Pas de codes erreur négatifs
    (F.col("consumption_mw") < 10)           # Pas d'outliers
)

df_clean.write.mode("overwrite").format("delta").saveAsTable("Lakehouse_silver.silver.consumption_clean")
print(f"✅ Table silver.consumption_clean créée : {df_clean.count()} lignes")

**Intended**:

- **Released**:`16426`lines
- **Deleted**: **854** = 525 NULL + 305 negatives + 24 outliers

---
## Cell 4: Cleaning check (report)

In [ ]:
# Compter les lignes
original_count = spark.sql("SELECT COUNT(*) FROM Lakehouse_bronze.bronze.consumption_raw").collect()[0][0]
clean_count = spark.table("Lakehouse_silver.silver.consumption_clean").count()
removed_count = original_count - clean_count

print("="*50)
print("📊 VÉRIFICATION DU NETTOYAGE")
print("="*50)
print(f"Lignes originales   : {original_count:,}")
print(f"Lignes nettoyées    : {clean_count:,}")
print(f"Lignes supprimées   : {removed_count:,} ({removed_count/original_count*100:.1f}%)")
print("="*50)

**Intended**:

- **Originals**: 18 144 · **cleaned**: 16 426
- **Deleted**: **1,718 (9.5%)**
- **Coherence**: same number as lines with anomaly in the Bronze report

---
## Cell 5: Standardization of date formats (`consumption_dates_fixed`)

- **`F.coalesce()`**: test each format **in order**, keep first non-NULL result
- **Result**: single column`timestamp_clean` de type `TIMESTAMP`

| Entry | Essai 1 (ISO) | Essai 2 (`dd/MM/yyyy`) | Result |
| --- | --- | --- | --- |
| `2025-01-22 01:00:00` | ✅ | — | 2025-01-22 01:00:00 |
| `17/01/2025 18:15:00` | NULL | ✅ | 2025-01-17 18:15:00 |

In [ ]:
from pyspark.sql import functions as F

df = spark.table("Lakehouse_silver.silver.consumption_clean")

# Normalisation formats de dates
df = df.withColumn(
    "timestamp_clean",
    F.coalesce(
        F.to_timestamp(F.col("timestamp")),                            # ISO 8601
        F.to_timestamp(F.col("timestamp"), "dd/MM/yyyy HH:mm:ss"),     # Français
        F.to_timestamp(F.col("timestamp"), "yyyy-MM-dd HH:mm:ss")      # Américain
    )
)

# Conserver les autres colonnes
df = df.select("site_id", "consumption_mw", "voltage_v", "frequency_hz", "status", "timestamp_clean")

df.write.mode("overwrite").format("delta").saveAsTable("Lakehouse_silver.silver.consumption_dates_fixed")
print(f"✅ Table silver.consumption_dates_fixed créée : {df.count()} lignes")
print("Dates non converties (NULL) :", df.filter(F.col("timestamp_clean").isNull()).count())

**Intended**:

- **Lines**:`16426`
- **Dates non converties** : **0**
- **Colonnes** : `timestamp`replaced by`timestamp_clean`

---
## Cell 6: Hourly aggregation (`consumption_hourly`)

- **Aggregation**: 4 measurements of 15 min → **1 line per hour per site**
- **Colonnes** : moyenne, max, min, nombre de mesures (`measurements`)

In [ ]:
from pyspark.sql import functions as F

df = spark.table("Lakehouse_silver.silver.consumption_dates_fixed").filter(
    F.col("timestamp_clean").isNotNull()
)

df_hourly = df.groupBy(
    F.date_trunc("hour", "timestamp_clean").alias("hour"),
    "site_id"
).agg(
    F.avg("consumption_mw").alias("avg_consumption_mw"),
    F.max("consumption_mw").alias("max_consumption_mw"),
    F.min("consumption_mw").alias("min_consumption_mw"),
    F.count("*").alias("measurements")
)

df_hourly.write.mode("overwrite").format("delta").saveAsTable("Lakehouse_silver.silver.consumption_hourly")
print(f"✅ Table silver.consumption_hourly créée : {df_hourly.count()} lignes")
df_hourly.groupBy("measurements").count().orderBy("measurements").show()

**Intended**:

- **Lines**:`4320`= 6 sites × 30 days × 24 hours (all hours covered)
- **`measurements`** : **4** → 3,533 lines · **3** → 721 · **2** → 65 · **1** → 1
- **Partial hours** due to lines removed from cleaning

---
## Cell 7: Joint market prices (`consumption_with_prices`)

- **Train**:`market_prices`is in step of **15 min** (2,880 lines)
- **Direct connection**: would multiply lines by 4 (17,280)
- **Solution**: first calculate the **hourly average price**

In [ ]:
from pyspark.sql import functions as F

df_conso = spark.table("Lakehouse_silver.silver.consumption_hourly")
df_prix = spark.sql("SELECT * FROM Lakehouse_bronze.bronze.market_prices")

# Prix au pas de 15 min -> prix moyen horaire (1 ligne par heure)
df_prix = (
    df_prix
    .withColumn("hour", F.date_trunc("hour", F.col("timestamp").cast("timestamp")))
    .groupBy("hour")
    .agg(
        F.round(F.avg("price_eur_mwh"), 2).alias("price_eur_mwh"),
        F.first("market").alias("market")
    )
)

# Jointure
df_with_prices = df_conso.join(df_prix, on="hour", how="left")

df_with_prices.write.mode("overwrite").format("delta").saveAsTable("Lakehouse_silver.silver.consumption_with_prices")
print(f"✅ Table silver.consumption_with_prices créée : {df_with_prices.count()} lignes")
print("Prix manquants :", df_with_prices.filter(F.col("price_eur_mwh").isNull()).count())
df_with_prices.show(20)

**Intended**:

- **Lines**:`4320`· **missing prices** : **0**
- **Colonnes** : 8 (`hour`, `site_id`, `avg/max/min_consumption_mw`, `measurements`, `price_eur_mwh`, `market`)
- **Price**: identical for all sites of the same time
- **LEFT JOIN** : aucune heure de consommation perdue

---
## Cell 8: Verification and Overview of`consumption_with_prices`

In [ ]:
# La table silver.consumption_with_prices existe déjà (créée en cellule 7)
df = spark.table("Lakehouse_silver.silver.consumption_with_prices")
print(f"✅ Table silver.consumption_with_prices sauvegardée : {df.count()} lignes")
print("\n📊 Aperçu des données nettoyées :")
df.show(10)
print("\n➡️ Prochaine étape : enrichissement avancé (cellule 9)")

**Intended**:

- **Lines**:`4320`
- **Overview**: 10 lines, **Stable entry point** for enrichment

---
## Cell 9: Advanced enrichment (`consumption_enriched`)

Window functions (*window functions*) and joins to create analysis and ML variables:

- **`baseline_7d_mw`**: slippery average of the previous 7 days, by site
- **`ratio_vs_baseline`** : consommation ÷ baseline
- **`z_score` / `anomaly`**: deviation from the average site in standard deviations; anomaly if |z| > 3
- **`hour_of_day`, `day_of_week`, `is_weekend`** : variables temporelles
- **Meetings**: reference`sites_reference`and weather`weather_data`
- **`lag_24h`, `lag_168h`**: consumption at J-1 and J-7 (same time)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# ─── 1. Charger la base ───────────────────────────────────────────────
df = spark.table("Lakehouse_silver.silver.consumption_with_prices")

# ─── 2. Baseline 7 jours glissants ───────────────────────────────────
w7 = Window.partitionBy("site_id") \
           .orderBy(F.col("hour").cast("long")) \
           .rangeBetween(-7 * 24 * 3600, -1)

df = df.withColumn("baseline_7d_mw", F.avg("avg_consumption_mw").over(w7))

# ─── 3. Ratio vs baseline ─────────────────────────────────────────────
df = df.withColumn(
    "ratio_vs_baseline",
    F.when(F.col("baseline_7d_mw") > 0,
           F.col("avg_consumption_mw") / F.col("baseline_7d_mw")
    ).otherwise(None)
)

# ─── 4. Z-score et détection anomalies ───────────────────────────────
w_site = Window.partitionBy("site_id")

df = df.withColumn("mean_site", F.avg("avg_consumption_mw").over(w_site)) \
       .withColumn("std_site",  F.stddev("avg_consumption_mw").over(w_site)) \
       .withColumn("z_score",
           F.when(F.col("std_site") > 0,
               (F.col("avg_consumption_mw") - F.col("mean_site")) / F.col("std_site")
           ).otherwise(0)
       ) \
       .withColumn("anomaly",
           F.when(F.abs(F.col("z_score")) > 3, "⚠️ ANOMALIE").otherwise("✅ Normal")
       ) \
       .drop("mean_site", "std_site")

# ─── 5. Features temporelles ─────────────────────────────────────────
df = df.withColumn("hour_of_day", F.hour("hour")) \
       .withColumn("day_of_week", F.dayofweek("hour")) \
       .withColumn("is_weekend",  F.dayofweek("hour").isin(1, 7))

# ─── 6. Jointure référentiel sites ───────────────────────────────────
df_sites = spark.sql("SELECT * FROM Lakehouse_bronze.bronze.sites_reference")
df = df.join(df_sites, on="site_id", how="left")

# ─── 7. Jointure météo (globale, pas de site_id) ─────────────────────
df_weather = spark.sql("SELECT * FROM Lakehouse_bronze.bronze.weather_data") \
                  .withColumn("hour_w", F.date_trunc("hour", F.col("timestamp").cast("timestamp"))) \
                  .select("hour_w", "temperature_c", "wind_speed_ms")

df = df.join(df_weather, df["hour"] == df_weather["hour_w"], how="left") \
       .drop("hour_w")

# ─── 8. Flag maintenance (placeholder : False par défaut) ─────────────
df = df.withColumn("in_maintenance", F.lit(False))

# ─── 9. Features ML : lag J-1 et J-7 ────────────────────────────────
w_lag = Window.partitionBy("site_id").orderBy("hour")

df = df.withColumn("lag_24h",  F.lag("avg_consumption_mw", 24).over(w_lag)) \
       .withColumn("lag_168h", F.lag("avg_consumption_mw", 168).over(w_lag))

# ─── 10. Colonne prediction (placeholder) ────────────────────────────
df = df.withColumn("prediction", F.col("baseline_7d_mw"))

# ─── 11. Sauvegarde ───────────────────────────────────────────────────
df.write.mode("overwrite").format("delta") \
  .saveAsTable("Lakehouse_silver.silver.consumption_enriched")

count = df.count()
print(f"✅ Table silver.consumption_enriched créée : {count} lignes")
df.printSchema()

**Intended**:

- **Lines**:`4320`
- **Colonnes** : **27**
  - 8 issues de `with_prices`
  - 7 derivatives (baseline, ratio, z-score, anomaly, 3 temporal variables)
  - 6 of the reference sites, 2 weather
  - `in_maintenance`, 2 lags, `prediction`

### Verification (add)

In [ ]:
from pyspark.sql import functions as F

df = spark.table("Lakehouse_silver.silver.consumption_enriched")

print("Anomalies (|z| > 3) :", df.filter(F.col("anomaly").contains("ANOMALIE")).count())
print("baseline_7d_mw NULL :", df.filter(F.col("baseline_7d_mw").isNull()).count())
print("lag_24h NULL        :", df.filter(F.col("lag_24h").isNull()).count())
print("lag_168h NULL       :", df.filter(F.col("lag_168h").isNull()).count())
print("temperature NULL    :", df.filter(F.col("temperature_c").isNull()).count())

df.groupBy("site_id", "site_type").agg(F.round(F.avg("avg_consumption_mw"), 3).alias("moy_mw")) \
  .orderBy("site_id").show()

**Intended**:

- **Anomalies (|z| > 3)** : 36
- **`baseline_7d_mw`NULL** : 6 (first hour of each site, no history)
- **`lag_24h`NULL**: 144 (first 24 hours × 6 sites) · **`lag_168h` NULL** : 1 008 (168 h × 6 sites)
- **`temperature_c` NULL** : 0
- **Consommation moyenne par site** : IND_001 ≈ 2,50 MW · IND_002 ≈ 1,79 · COM_001 ≈ 0,74 · COM_002 ≈ 0,56 · RES_001 ≈ 0,23 · RES_002 ≈ 0,17

---
## Cell 10: Model of prediction (`consumption_predictions`)

- **Formule** : `0,5 × baseline_7d + 0,3 × lag_24h + 0,2 × lag_168h`
- **Repli** : baseline seule si l'historique manque
- **Errors**: absolute (MW) and relative (%)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window

df = spark.table("Lakehouse_silver.silver.consumption_enriched")

# Fenêtre par site ordonnée par heure
w_lag = Window.partitionBy("site_id").orderBy("hour")

# Prédiction = moyenne pondérée baseline 7j + lag 24h + lag 168h
df_pred = df.withColumn("lag_24h",  F.lag("avg_consumption_mw", 24).over(w_lag)) \
            .withColumn("lag_168h", F.lag("avg_consumption_mw", 168).over(w_lag)) \
            .withColumn(
                "prediction",
                F.when(
                    F.col("lag_24h").isNotNull() & F.col("lag_168h").isNotNull(),
                    F.round(
                        F.col("baseline_7d_mw") * 0.5 +
                        F.col("lag_24h")         * 0.3 +
                        F.col("lag_168h")         * 0.2,
                        4
                    )
                ).otherwise(F.col("baseline_7d_mw"))  # fallback si pas assez d'historique
            ) \
            .withColumn(
                "prediction_error_mw",
                F.round(F.abs(F.col("avg_consumption_mw") - F.col("prediction")), 4)
            ) \
            .withColumn(
                "prediction_error_pct",
                F.when(
                    F.col("avg_consumption_mw") > 0,
                    F.round(F.col("prediction_error_mw") / F.col("avg_consumption_mw") * 100, 2)
                ).otherwise(None)
            )

# Sélection des colonnes utiles pour la table predictions
df_pred_final = df_pred.select(
    "hour", "site_id",
    "avg_consumption_mw",
    "prediction",
    "prediction_error_mw",
    "prediction_error_pct",
    "lag_24h",
    "lag_168h"
)

df_pred_final.write.mode("overwrite").format("delta") \
    .saveAsTable("Lakehouse_silver.silver.consumption_predictions")

print(f"✅ silver.consumption_predictions créée : {df_pred_final.count():,} lignes")
df_pred_final.show(5)

**Intended**:

- **Lines**:`4,320`· **columns**: 8
- **No prediction**: 6 lines (first hour of each site)

### Model evaluation (add)

In [ ]:
from pyspark.sql import functions as F

dfp = spark.table("Lakehouse_silver.silver.consumption_predictions").filter(F.col("prediction").isNotNull())

dfp.agg(
    F.round(F.avg("prediction_error_mw"), 3).alias("erreur_moy_mw"),
    F.round(F.avg("prediction_error_pct"), 1).alias("erreur_moy_pct"),
    F.round(F.percentile_approx("prediction_error_pct", 0.5), 1).alias("erreur_mediane_pct"),
    F.max("prediction_error_pct").alias("erreur_max_pct")
).show()

**Attended** (order of magnitude):

- **Average error**: 
- **Volunteerly simple model**: the error weighs heavily on **small sites** (residential)
- **Gold Track**: regression, hourly variables, weather

---
## Executive summary

| Table | Lines | Objective |
| --- | --- | --- |
| `consumption_dedup` | 17 280 | removal of duplicates |
| `consumption_clean` | 16 426 | filtering invalid values |
| `consumption_dates_fixed` | 16 426 | normalization of timestamps |
| `consumption_hourly` | 4 320 | aggregation 15 min → 1 h |
| `consumption_with_prices` | 4 320 | enrichissement prix EPEX |
| `consumption_enriched` | 4 320 | baseline, anomalies, weather, ML variables (27 columns) |
| `consumption_predictions` | 4 320 | weighted predictive model |

- **Concepts** : `DISTINCT`, `filter`, `coalesce`, `groupBy + agg`, `LEFT JOIN`
- ** Windows**: *window functions*, z-score, ratios, lags
- **Idempotence** : `overwrite`and`IF NOT EXISTS`allow to restart the notebook without error
- **Continued**: Silver tables feed the workshop **Gold**